# Current HMM portfolio experiments

Run and visualize the **80/20, 90/10, and 95/5 stock/cash experiments** using the
same frozen plans, simulator, and comparison checks as the terminal workflow.

| Testing window | Training window | Last testing session |
|---|---|---|
| 2023–2026 | 2015–2022 | October 8, 2026 |
| 2020–2026 | 2000–2019 | October 30, 2026 |

**Start here:** install `requirements-notebook.txt` into the repo-root `.venv`
using the [README instructions](README.md#run-and-visualize-in-jupyter), select
that environment as the notebook kernel, and run cells from top to bottom.
The first run defaults to **reference mode**: it plots committed results without
downloading prices or training. Other modes load your own completed runs or
train fresh experiments. Generated exports are saved under ignored `output/`.


In [ ]:
import json
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import numpy as np
import pandas as pd
from IPython.display import display

# Find the project whether Jupyter starts at the repo root or in this folder.
ROOT = None
for parent in (Path.cwd(), *Path.cwd().parents):
    for candidate in (parent, parent / "HiddenMarkovModel"):
        if (candidate / "portfolio_train2000_cash20_plan.json").is_file():
            ROOT = candidate.resolve()
            break
    if ROOT is not None:
        break
if ROOT is None:
    raise FileNotFoundError("Start Jupyter from q-bot or its HiddenMarkovModel folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from compare_portfolios import STUDIES, build

get_ipython().run_line_magic("matplotlib", "inline")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
print(f"Kernel Python: {sys.executable}")
print(f"Experiment folder: {ROOT}")


## Choose what to run

- `MODE = "reference"`: use the saved reference CSVs. No training or downloads.
- `MODE = "existing"`: read completed books and rebuild the checked comparisons
  in a new session folder. Default input paths come from the plans. Add any
  timestamped manual folders to `EXISTING_RUN_DIRS`; unspecified allocations
  keep their plan defaults. All three allocations must be complete per window.
- `MODE = "fresh"`: train all three allocations for each selected window in new
  folders. Both cost versions and all plan-declared evaluation windows run.
  Set `DOWNLOAD_INPUTS = True` to fetch the required inputs first. Otherwise,
  existing audited caches are reused and verified by the runner.

`WINDOWS` can contain `"2023"`, `"2020"`, or both. `PLOT_VERSION` controls the
figures; summary CSVs and tables always keep **both** cost versions. Change the
settings here, then rerun this cell and every cell below it. A new session path
prevents a repeated fresh run from overwriting locked protocols.


In [ ]:
MODE = "reference"                 # "reference", "existing", or "fresh"
WINDOWS = ["2023", "2020"]
PLOT_VERSION = "half_cent_slippage" # or "zero_slippage"
DOWNLOAD_INPUTS = False            # only used in fresh mode

# Optional overrides for MODE = "existing"; paths are relative to ROOT or absolute.
# Example (use your actual timestamps; complete the other allocations as well):
# EXISTING_RUN_DIRS = {"2023": {
#     "80/20": "output/manual_2023_80_20_20261103_010320",
#     "90/10": "output/manual_2023_90_10_YOUR_TIMESTAMP",
#     "95/5": "output/manual_2023_95_5_YOUR_TIMESTAMP",
# }}
EXISTING_RUN_DIRS = {}

SESSION_DIR = ROOT / "output" / ("notebook_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f_UTC"))
ALLOCATIONS = ["80/20", "90/10", "95/5"]
VERSIONS = ["zero_slippage", "half_cent_slippage"]
if MODE not in {"reference", "existing", "fresh"}:
    raise ValueError("Choose reference, existing, or fresh mode.")
if not WINDOWS or len(set(WINDOWS)) != len(WINDOWS) or set(WINDOWS) - STUDIES.keys():
    raise ValueError("WINDOWS must contain 2023, 2020, or both, without duplicates.")
if PLOT_VERSION not in VERSIONS:
    raise ValueError(f"PLOT_VERSION must be one of {VERSIONS}.")
if set(EXISTING_RUN_DIRS) - STUDIES.keys():
    raise ValueError("Existing-run window keys must be 2023 or 2020.")
for window, overrides in EXISTING_RUN_DIRS.items():
    if set(overrides) - set(ALLOCATIONS):
        raise ValueError(f"Unknown allocation override for {window}.")
print(f"Mode: {MODE}; windows: {WINDOWS}; plotted cost version: {PLOT_VERSION}")
print(f"Exports: {SESSION_DIR}")


## Inspect the frozen plans

The stock percentages are **minimum exposure after the initial holding**, with
cash capped at 20%, 10%, or 5% at daily opens. Books start with $1 million fully
invested for the first 20 sessions. These are not exact continuously rebalanced
stock/cash ratios. This notebook uses the checked-in settings unchanged.


In [ ]:
plans = {}
plan_rows = []
for window in WINDOWS:
    names, _ = STUDIES[window]
    plans[window] = {label: json.loads((ROOT / name).read_text(encoding="utf-8"))
                     for label, name in zip(ALLOCATIONS, names)}
    for label, name in zip(ALLOCATIONS, names):
        plan = plans[window][label]
        plan_rows.append({"window": window, "allocation": label, "plan": name,
                          "training_start": plan["training_start"],
                          "testing_start": plan["validation_start"], "testing_end": plan["end"],
                          "initial_equity": plan["configuration"]["initial_equity"],
                          "max_cash_weight": plan["configuration"]["max_cash_weight"]})
display(pd.DataFrame(plan_rows))


## Download / run / verify (controlled by MODE)

Fresh runs can take time. Progress is streamed below. Download or training errors
stop execution; missing runs are never replaced silently with reference data.
The comparison builder checks plan/source locks, matched benchmarks, daily stock
floors, portfolio accounting, and reconciliation of full-period Sharpe to daily
books. Comparisons go into this session's folder, not the committed references.


In [ ]:
def run_script(script, *arguments):
    command = [sys.executable, "-u", str(ROOT / script), *map(str, arguments)]
    print("Running:", subprocess.list2cmdline(command), flush=True)
    with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        status = process.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)

comparison_dirs = {}
if MODE == "reference":
    comparison_dirs = {window: ROOT / "results" / STUDIES[window][1] for window in WINDOWS}
    print("Using committed reference comparisons; no data download or training.")
else:
    if MODE == "fresh":
        if DOWNLOAD_INPUTS:
            data_plans = {"2023": "portfolio_study_plan.json",
                          "2020": "portfolio_train2000_cash20_plan.json"}
            for window in WINDOWS:
                run_script("download_portfolio_study_data.py", "--plan", data_plans[window])
        missing_inputs = [ROOT / plans[window]["80/20"]["data_dir"] / "manifest.json"
                          for window in WINDOWS
                          if not (ROOT / plans[window]["80/20"]["data_dir"] / "manifest.json").is_file()]
        if missing_inputs:
            raise FileNotFoundError("Missing data manifests. Set DOWNLOAD_INPUTS = True and rerun: "
                                    + ", ".join(map(str, missing_inputs)))
    for window in WINDOWS:
        if MODE == "fresh":
            runs = {label: SESSION_DIR / window / label.replace("/", "_") for label in ALLOCATIONS}
            for label, name in zip(ALLOCATIONS, STUDIES[window][0]):
                run_script("run_portfolio_study.py", "--plan", name, "--output-dir", runs[label])
        else:
            overrides = EXISTING_RUN_DIRS.get(window, {})
            runs = {label: ROOT / overrides.get(label, plans[window][label]["output_dir"])
                    for label in ALLOCATIONS}
        missing = [str(folder / name) for folder in runs.values()
                   for name in ("metrics.csv", "research_report.json", "protocol.json", "REPORT.md")
                   if not (folder / name).is_file()]
        if missing:
            raise FileNotFoundError("Complete all three allocation runs or fix EXISTING_RUN_DIRS. Missing: "
                                    + ", ".join(missing))
        comparison_dirs[window] = SESSION_DIR / window / "comparison"
        build(window, run_dirs=runs, output_dir=comparison_dirs[window])
print("Comparison inputs:")
for window, folder in comparison_dirs.items():
    print(window, folder)


## Metrics and yearly Sharpe summaries

Full-period Sharpe is computed over the continuous book. **Mean yearly Sharpe**
is the arithmetic mean of calendar-year Sharpes; its SD is the **sample** standard
deviation across years (`ddof=1`). The partial 2026 year is included, so these
summaries have four yearly observations for the 2023 window and seven for 2020.
Undefined yearly Sharpes are excluded and counted, rather than replaced with zero.
The SD describes year-to-year variation; it is not a standard error or confidence
interval. Return and drawdown columns below are formatted as percentages.


In [ ]:
summaries, yearly_tables, equity_tables = {}, {}, {}
renames = {f"{label} stocks/cash": label for label in ALLOCATIONS}
BENCHMARKS = ["IWV (Russell 3000 ETF proxy)", "100-stock buy and hold"]
LABELS = ALLOCATIONS + BENCHMARKS
for window, folder in comparison_dirs.items():
    summary = pd.read_csv(folder / "comparison.csv")
    yearly = pd.read_csv(folder / "yearly_sharpe.csv")
    summaries[window], yearly_tables[window] = summary, yearly
    initial_equity = plans[window]["80/20"]["configuration"]["initial_equity"]
    for version in VERSIONS:
        curves = pd.read_csv(folder / f"equity_{version}.csv", index_col="Date", parse_dates=True).rename(columns=renames)
        if (list(curves.columns) != LABELS or curves.empty or curves.index.has_duplicates
                or not curves.index.is_monotonic_increasing or not np.isfinite(curves.to_numpy()).all()):
            raise ValueError(f"Invalid or incomplete equity curves: {window}, {version}")
        metrics = summary.loc[summary.version.eq(version)].set_index("allocation").loc[LABELS]
        np.testing.assert_allclose(curves.iloc[-1], metrics.final_equity, rtol=0, atol=1e-6)
        np.testing.assert_allclose(curves.iloc[-1] / initial_equity - 1,
                                   metrics.net_cumulative_return, rtol=0, atol=1e-10)
        equity_tables[window, version] = curves
    columns = ["allocation", "version", "net_cumulative_return", "annualized_return",
               "max_drawdown", "sharpe", "annual_sharpe_mean", "annual_sharpe_std",
               "annual_sharpe_count", "annual_sharpe_undefined_years", "matches_stock_floor"]
    print(f"Testing window: {window} through {plans[window]['80/20']['end']}")
    display(summary[columns].style.format({"net_cumulative_return": "{:+.2%}",
            "annualized_return": "{:+.2%}", "max_drawdown": "{:.2%}",
            "sharpe": "{:.3f}", "annual_sharpe_mean": "{:.3f}", "annual_sharpe_std": "{:.3f}"}))


## Allocation and benchmark charts

Each testing window is plotted separately using the selected cost version.
The return difference is measured in **percentage points relative to 80/20**,
using the same starting capital. Drawdown includes starting capital before entry
costs. Annual Sharpe bars preserve the continuous book across year boundaries.
IWV is the investable Russell 3000 ETF proxy; the passive control holds the same
100-stock basket. Both use the matching cost version.


In [ ]:
COLORS = dict(zip(LABELS, ["#0072B2", "#D55E00", "#009E73", "#9B59B6", "#52525B"]))
for window in WINDOWS:
    curves = equity_tables[window, PLOT_VERSION]
    initial = plans[window]["80/20"]["configuration"]["initial_equity"]
    annual = yearly_tables[window]
    annual = annual.loc[annual.version.eq(PLOT_VERSION)].pivot(index="year", columns="allocation", values="sharpe").reindex(columns=LABELS)
    fig, axes = plt.subplots(2, 2, figsize=(16, 10), layout="constrained")
    for label in LABELS:
        style = "--" if label in BENCHMARKS else "-"
        axes[0, 0].plot(curves.index, curves[label], color=COLORS[label], linestyle=style, label=label)
        axes[0, 1].plot(curves.index, (curves[label] - curves["80/20"]) / initial * 100,
                        color=COLORS[label], linestyle=style)
        drawdown = (curves[label] / curves[label].cummax().clip(lower=initial) - 1) * 100
        axes[1, 0].plot(curves.index, drawdown, color=COLORS[label], linestyle=style)
    axes[0, 0].set(title="Portfolio equity after costs", ylabel="Equity")
    axes[0, 0].yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"${value / 1e6:.2f}m"))
    axes[0, 0].legend(fontsize=8)
    axes[0, 1].set(title="Cumulative return difference versus 80/20", ylabel="Percentage points")
    axes[0, 1].axhline(0, color="#A1A1AA", linewidth=0.7)
    axes[1, 0].set(title="Drawdown from running peak", ylabel="Drawdown (%)")
    annual.plot.bar(ax=axes[1, 1], color=[COLORS[label] for label in LABELS], rot=0, legend=False)
    axes[1, 1].set(title="Calendar-year Sharpe (partial 2026 included)", ylabel="Annualized Sharpe", xlabel="Calendar year")
    axes[1, 1].axhline(0, color="#A1A1AA", linewidth=0.7)
    for axis in axes.flat:
        axis.grid(axis="y", color="#E4E4E7", linewidth=0.6)
    fee = "$0.005/share per execution" if PLOT_VERSION == "half_cent_slippage" else "zero slippage"
    fig.suptitle(f"{window} testing window | {fee} | {MODE} results", fontsize=16)
    folder = SESSION_DIR / window / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    for extension in ("png", "svg"):
        fig.savefig(folder / f"comparison_{PLOT_VERSION}.{extension}", dpi=160, facecolor="white")
    plt.show()
    plt.close(fig)
    print(f"Charts saved to {folder}")


## Export tables and interpret the results

The tables below report whether each model's continuous equity curve matches
its mandatory stock-floor control. A match means compulsory stock exposure
explains that run's performance; it does not demonstrate incremental HMM skill.
Increasing exposure can raise both returns and drawdowns.

The selected current stock universe has survivorship/selection bias, and these
historical periods have already been studied. Results are retrospective research,
not independent evidence of a trading advantage. See [methodology](docs/METHODOLOGY.md).


In [ ]:
for window in WINDOWS:
    folder = SESSION_DIR / window / "tables"
    folder.mkdir(parents=True, exist_ok=True)
    summaries[window].to_csv(folder / "comparison.csv", index=False)
    yearly_tables[window].to_csv(folder / "yearly_sharpe.csv", index=False)
    for version in VERSIONS:
        equity_tables[window, version].to_csv(folder / f"equity_{version}.csv", index_label="Date")
    model = summaries[window].loc[summaries[window].strategy.eq("hmm"),
              ["allocation", "version", "matches_stock_floor", "return_vs_80_20_pp"]]
    print(f"{window}: model versus matched stock-floor control")
    display(model)
print(f"All notebook exports: {SESSION_DIR}")
